# 07 — DPO Exploration

Explores the DPO alignment pipeline — preference dataset analysis, reward margin tracking, training dynamics, and before/after alignment comparison.

Run after `make prepare-dpo` and `make dpo`.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import math
import random
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import torch

DATA_DIR = Path('../data')
RESULTS_DIR = Path('../results')
DPO_DIR = DATA_DIR / 'dpo'

rng = random.Random(42)

## 1. Dataset overview

In [ ]:
stats_path = DPO_DIR / 'stats.json'
train_path = DPO_DIR / 'train.jsonl'
val_path = DPO_DIR / 'val.jsonl'

if stats_path.exists():
    with open(stats_path) as f:
        stats = json.load(f)

    print(f"Total pairs:  {stats['total']:,}")
    print(f"Train:        {stats['train']:,}")
    print(f"Val:          {stats['val']:,}")
    print()
    print("Source breakdown:")
    for source, count in stats['sources'].items():
        pct = 100 * count / stats['total']
        print(f"  {source:<20} {count:>8,}  ({pct:.1f}%)")
else:
    print("DPO stats not found — run: python alignment/data/prepare_dpo.py")

## 2. Source distribution

In [ ]:
if stats_path.exists():
    fig, ax = plt.subplots(figsize=(7, 5))
    sources = list(stats['sources'].keys())
    counts = list(stats['sources'].values())
    colors = ['#4C72B0', '#DD8452', '#55A868']

    wedges, texts, autotexts = ax.pie(
        counts, labels=sources, autopct='%1.1f%%',
        colors=colors, startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
    )
    for t in autotexts:
        t.set_fontsize(10)
    ax.set_title('DPO Dataset — Source Distribution', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. Load sample preference pairs

In [ ]:
SAMPLE_SIZE = 3000
records = []

if train_path.exists():
    with open(train_path) as f:
        for i, line in enumerate(f):
            if i >= SAMPLE_SIZE:
                break
            records.append(json.loads(line))
    print(f"Loaded {len(records):,} records")
else:
    print("DPO training data not found — run: python alignment/data/prepare_dpo.py")

## 4. Response length analysis — chosen vs rejected

In [ ]:
if records:
    chosen_lens = [len(r['chosen']) for r in records]
    rejected_lens = [len(r['rejected']) for r in records]
    prompt_lens = [len(r['prompt']) for r in records]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(chosen_lens, bins=50, color='#55A868', alpha=0.8, edgecolor='white', label='chosen')
    axes[0].hist(rejected_lens, bins=50, color='#C44E52', alpha=0.6, edgecolor='white', label='rejected')
    axes[0].set_xlabel('Characters')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Response Length — Chosen vs Rejected', fontsize=11, fontweight='bold')
    axes[0].legend()
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

    # Length difference
    diffs = [c - r for c, r in zip(chosen_lens, rejected_lens)]
    axes[1].hist(diffs, bins=50, color='#4C72B0', alpha=0.85, edgecolor='white')
    axes[1].axvline(0, color='red', linestyle='--', alpha=0.7)
    axes[1].axvline(np.mean(diffs), color='black', linestyle='--', label=f'mean={np.mean(diffs):.0f}')
    axes[1].set_xlabel('chosen length - rejected length')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Length Difference (chosen - rejected)', fontsize=11, fontweight='bold')
    axes[1].legend(fontsize=9)

    axes[2].hist(prompt_lens, bins=50, color='#8172B2', alpha=0.85, edgecolor='white')
    axes[2].axvline(512, color='red', linestyle='--', label='max_prompt_length=512')
    axes[2].set_xlabel('Characters')
    axes[2].set_ylabel('Count')
    axes[2].set_title('Prompt Length Distribution', fontsize=11, fontweight='bold')
    axes[2].legend(fontsize=9)

    plt.tight_layout()
    plt.show()

    print(f"Chosen length   — mean: {np.mean(chosen_lens):.0f}, median: {np.median(chosen_lens):.0f}")
    print(f"Rejected length — mean: {np.mean(rejected_lens):.0f}, median: {np.median(rejected_lens):.0f}")
    print(f"Length diff     — mean: {np.mean(diffs):.0f} (positive = chosen longer)")
    chosen_longer = sum(1 for d in diffs if d > 0)
    print(f"Chosen longer:    {chosen_longer:,} ({100*chosen_longer/len(diffs):.1f}%)")

## 5. Sample preference pairs per source

In [ ]:
if records:
    sources = ['hh-rlhf', 'orca', 'argilla']
    for source in sources:
        source_records = [r for r in records if r.get('source') == source]
        if not source_records:
            continue
        sample = rng.choice(source_records)

        print(f"{'='*65}")
        print(f"SOURCE: {source.upper()}")
        print(f"{'='*65}")
        print(f"PROMPT (last 200 chars):")
        print(f"  ...{sample['prompt'][-200:]}")
        print(f"\nCHOSEN ({len(sample['chosen'])} chars):")
        print(f"  {sample['chosen'][:300]}{'...' if len(sample['chosen']) > 300 else ''}")
        print(f"\nREJECTED ({len(sample['rejected'])} chars):")
        print(f"  {sample['rejected'][:300]}{'...' if len(sample['rejected']) > 300 else ''}")
        print()

## 6. DPO training curves

In [ ]:
dpo_dir = RESULTS_DIR / 'slm-125m-dpo'
state_files = list(dpo_dir.glob('**/trainer_state.json'))

if state_files:
    with open(sorted(state_files)[-1]) as f:
        state = json.load(f)

    logs = state.get('log_history', [])
    train_logs = [x for x in logs if 'loss' in x and 'eval_loss' not in x]
    eval_logs = [x for x in logs if 'eval_loss' in x]

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    # Loss
    if train_logs:
        steps = [x['step'] for x in train_logs]
        losses = [x['loss'] for x in train_logs]
        axes[0, 0].plot(steps, losses, color='#4C72B0', alpha=0.5, linewidth=0.8)
        if len(losses) > 10:
            smoothed = np.convolve(losses, np.ones(10)/10, mode='valid')
            axes[0, 0].plot(steps[5:-4], smoothed, color='#4C72B0', linewidth=2, label='train')
    if eval_logs:
        eval_steps = [x['step'] for x in eval_logs]
        eval_losses = [x['eval_loss'] for x in eval_logs]
        axes[0, 0].plot(eval_steps, eval_losses, color='#C44E52', linewidth=2,
                        marker='o', markersize=4, label='eval')
    axes[0, 0].set_title('DPO Loss', fontsize=11, fontweight='bold')
    axes[0, 0].set_xlabel('Step')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend(fontsize=9)

    # Reward margin — rewards/chosen - rewards/rejected
    chosen_rewards = [x.get('rewards/chosen', None) for x in train_logs]
    rejected_rewards = [x.get('rewards/rejected', None) for x in train_logs]
    if any(r is not None for r in chosen_rewards):
        margins = [c - r for c, r in zip(chosen_rewards, rejected_rewards)
                   if c is not None and r is not None]
        margin_steps = [s for s, c in zip(steps, chosen_rewards) if c is not None]
        axes[0, 1].plot(margin_steps, margins, color='#55A868', linewidth=1.5)
        axes[0, 1].axhline(0, color='red', linestyle='--', alpha=0.5)
        axes[0, 1].set_title('Reward Margin (chosen - rejected)', fontsize=11, fontweight='bold')
        axes[0, 1].set_xlabel('Step')
        axes[0, 1].set_ylabel('Margin')
    else:
        axes[0, 1].text(0.5, 0.5, 'Reward margins\nnot logged',
                        ha='center', va='center', transform=axes[0, 1].transAxes)

    # LR
    lrs = [x.get('learning_rate', 0) for x in train_logs]
    axes[1, 0].plot(steps, lrs, color='#8172B2', linewidth=1.5)
    axes[1, 0].set_title('Learning Rate', fontsize=11, fontweight='bold')
    axes[1, 0].set_xlabel('Step')
    axes[1, 0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0e}'))

    # Grad norm
    grad_norms = [x.get('grad_norm', 0) for x in train_logs]
    axes[1, 1].plot(steps, grad_norms, color='#C44E52', alpha=0.6, linewidth=0.8)
    axes[1, 1].axhline(1.0, color='black', linestyle='--', alpha=0.4, label='clip=1.0')
    axes[1, 1].set_title('Gradient Norm', fontsize=11, fontweight='bold')
    axes[1, 1].set_xlabel('Step')
    axes[1, 1].legend(fontsize=9)

    fig.suptitle('slm-125m DPO Training Dynamics', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No DPO checkpoints found — run: make dpo")

## 7. Before vs after DPO — generation comparison

In [ ]:
sft_dir = RESULTS_DIR / 'slm-125m-chat-code' / 'final'
dpo_dir_final = RESULTS_DIR / 'slm-125m-dpo' / 'final'

if sft_dir.exists() and dpo_dir_final.exists():
    from transformers import AutoConfig
    from model import SLMConfig, SLMForCausalLM
    from tokenizers import Tokenizer

    AutoConfig.register('slm', SLMConfig)
    tokenizer = Tokenizer.from_file(str(DATA_DIR / 'tokenizer' / 'slm_tokenizer.json'))

    sft_model = SLMForCausalLM.from_pretrained(str(sft_dir)).eval()
    dpo_model = SLMForCausalLM.from_pretrained(str(dpo_dir_final)).eval()

    def generate(model, prompt, max_new_tokens=150):
        input_ids = torch.tensor([tokenizer.encode(prompt).ids])
        with torch.no_grad():
            output = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=0,
                eos_token_id=3,
            )
        return tokenizer.decode(output[0].tolist(), skip_special_tokens=True)

    prompts = [
        (
            "Helpfulness",
            "<|system|>You are a helpful assistant.<|endofturn|>"
            "<|user|>Can you help me understand the difference between DPO and PPO for LLM alignment?<|endofturn|>"
            "<|assistant|>"
        ),
        (
            "Harmlessness",
            "<|system|>You are a helpful assistant.<|endofturn|>"
            "<|user|>How can I make my code more efficient?<|endofturn|>"
            "<|assistant|>"
        ),
    ]

    for label, prompt in prompts:
        print(f"{'='*65}")
        print(f"PROMPT TYPE: {label}")
        print(f"{'='*65}")
        print(f"\nSFT MODEL:")
        print(generate(sft_model, prompt))
        print(f"\nDPO MODEL:")
        print(generate(dpo_model, prompt))
        print()
else:
    print("Models not found — run: make sft && make dpo")

## 8. Beta sensitivity analysis

In [ ]:
# Visualize how beta affects the DPO loss landscape
betas = [0.01, 0.05, 0.1, 0.2, 0.5]
reward_margins = np.linspace(-3, 3, 200)

fig, ax = plt.subplots(figsize=(10, 4))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(betas)))

for beta, color in zip(betas, colors):
    # DPO loss as a function of reward margin
    loss = -np.log(1 / (1 + np.exp(-beta * reward_margins)))
    ax.plot(reward_margins, loss, label=f'β={beta}', color=color, linewidth=2)

ax.axvline(0, color='black', linestyle='--', alpha=0.3)
ax.set_xlabel('Reward margin (chosen - rejected)')
ax.set_ylabel('DPO loss')
ax.set_title('DPO Loss vs Reward Margin for Different β Values', fontsize=12, fontweight='bold')
ax.legend(title='Beta', fontsize=9)
ax.set_ylim(0, 2)
plt.tight_layout()
plt.show()

print("Higher β = steeper loss = stronger preference signal = more deviation from SFT reference")
print("Lower  β = flatter loss = weaker preference signal = stays closer to SFT reference")
print("Default: β=0.1")